In [30]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np

LOG.propagate = False
from uuid import uuid4
uuid4()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


UUID('5b7a0969-1383-4117-a2f7-5c54ec7f3457')

In [52]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.connect()

2026-05-04 15:24:35,606 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:d1:79:ab:49
2026-05-04 15:24:35,606 | INFO     |: Scanning for device with address: c0:42:d1:79:ab:49, service UUID: 4843fd62-068c-4a37-af34-e724c6a05681
2026-05-04 15:24:45,741 | INFO     |: Found 89 total devices
2026-05-04 15:24:45,742 | INFO     |: Found matching device: C0:42:D1:79:AB:49 (name: Artemis BLE)
2026-05-04 15:24:47,814 | INFO     |: Connected to c0:42:d1:79:ab:49


In [32]:
# ── CELL: Imports, Commands & Notification Handler ───────────────────────────
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import medfilt
from enum import Enum

class CommandTypes(Enum):
    PING              = 0
    SEND_TWO_INTS     = 1
    SEND_THREE_FLOATS = 2
    ECHO              = 3
    DUMP_LOGS         = 4
    GET_TIME_MILLIS   = 5
    GET_IMU_DATA      = 6
    START_LOGGING     = 7
    START_RUN         = 8
    GET_RUN_DATA      = 9
    START_PID         = 10
    SET_GAINS         = 11
    STOP_ROBOT        = 12
    SYSID_DROP        = 13   # free-fall from balance — extracts alpha1 & alpha2

print("Commands ready.")

# ── Notification Handler ─────────────────────────────────────────────────────
raw_packets = []

def notification_handler(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)
    raw_packets.append(s)

# Safety reset then subscribe
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except:
    pass

raw_packets.clear()
ble.start_notify(ble.uuid['RX_STRING'], notification_handler)
print("Subscribed. Listening for packets...")

Commands ready.
Subscribed. Listening for packets...


# Lab 12 — SysID & Kalman Filter Parameter Estimation

**Packet format** (each BLE notification):
```
T:<ms>|D1:<raw_pitch_deg>|D2:<kf_phi_or_pk>|MOT:<pwm>|ERR:<phi>|VEL:<kf_rate>|GYR:<raw_gyro_deg_s>
```

**Workflow:**
1. **Step A** — SysID Drop: hold car upright near `SETPOINT_DEG`, send `SYSID_DROP` (cmd 13), release. Fits `alpha1` (gravity).
2. **Step B** — Motor SysID: run a balancing session (`START_PID` cmd 10), retrieve the log. Fits `alpha2` (motor torque).
3. Paste the fitted values into the `.ino`:
   ```cpp
   const float KF_ALPHA1 = <alpha1>;
   const float KF_ALPHA2 = <alpha2>;
   ```

In [22]:
# ── Configuration — keep in sync with lab12_fixed.ino ────────────────────────
SETPOINT_DEG = 86.82   # SETPOINT_DEG in the .ino

## Step A — SysID Drop Test  →  fit alpha1 (gravity)

**Procedure:**
1. Hold the car upright near the balance point (~`SETPOINT_DEG`).
2. Run the cell below — it sends `SYSID_DROP` (cmd 13).
3. Release the car immediately. The robot logs for up to 3 s then idles.
4. Run the **retrieve** cell to pull the data.

In [46]:
# ── CELL: Trigger SysID Drop ─────────────────────────────────────────────────
raw_packets.clear()
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except: pass
ble.start_notify(ble.uuid['RX_STRING'], notification_handler)

print("Sending SYSID_DROP (cmd 13) — release the car NOW...")
ble.send_command(CommandTypes.SYSID_DROP, "")

Sending SYSID_DROP (cmd 13) — release the car NOW...


In [ ]:
# ── CELL: Wait, retrieve & parse SysID drop data ─────────────────────────────
import csv

# Wait for the robot to finish logging (3 s) then request data
time.sleep(4)
raw_packets.clear()
print("Requesting data (cmd 9)...")
ble.send_command(CommandTypes.GET_RUN_DATA, "")

# Poll until transfer completes OR packets go idle.
# The .ino streams packets on BLE TX and does NOT send a final "complete" BLE packet,
# so we use an idle-window fallback.
timeout_s = 45.0
idle_window_s = 5
t_start = time.time()
last_count = 0
last_change = t_start

while time.time() - t_start < timeout_s:
    n = len(raw_packets)

    # Future-proof: break if firmware ever sends an explicit completion packet.
    if n > 0 and "complete" in raw_packets[-1].lower():
        break

    if n != last_count:
        last_count = n
        last_change = time.time()
    elif n > 0 and (time.time() - last_change) >= idle_window_s:
        break

    time.sleep(0.2)

print(f"Received {len(raw_packets)} packets.")
if raw_packets:
    print("First:", raw_packets[0])
    print("Last: ", raw_packets[-1])

# ── Parse ─────────────────────────────────────────────────────────────────────
times_raw, pitch_raw, phi_raw, motor_log, gyro_raw = [], [], [], [], []

for line in raw_packets:
    try:
        # Expected format:
        # T:<ms>|D1:<raw_pitch>|D2:<kf_pitch_or_phi>|MOT:<pwm>|ERR:<phi>|VEL:<kf_rate>|GYR:<raw_gyro>
        parts = {}
        for p in line.strip().split('|'):
            if ':' not in p:
                continue
            k, v = p.split(':', 1)
            parts[k] = float(v)

        if 'T' not in parts:
            continue

        times_raw.append(parts['T'])
        pitch_raw.append(parts.get('D1', 0.0))   # absolute theta (deg)
        phi_raw.append(parts.get('ERR', 0.0))    # phi = theta - SETPOINT_DEG
        motor_log.append(parts.get('MOT', 0.0))  # PWM
        gyro_raw.append(parts.get('GYR', 0.0))   # raw gyro phi_dot (deg/s)
    except Exception:
        continue

if len(times_raw) == 0:
    print("No data received. Check BLE connection and that the robot ran.")
else:
    t = np.array([(x - times_raw[0]) / 1000.0 for x in times_raw])
    pitch = np.array(pitch_raw)
    phi = np.array(phi_raw)
    gyro = np.array(gyro_raw)   # phi_dot measured directly
    mot = np.array(motor_log)

    # Save CSV
    
    csv_path = "sysid_drop_data.csv"
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['time_s', 'pitch_deg', 'phi_deg', 'gyro_deg_s', 'motor_pwm'])
        for row in zip(t, pitch, phi, gyro, mot):
            writer.writerow(row)
    print(f"Saved {len(t)} samples → {csv_path}")

In [ ]:
# ── CELL: Fit alpha1 and plot drop test ──────────────────────────────────────
#
# Model (motors off):  phi_ddot = alpha1 * phi
# Regression:          phi_ddot vs phi  →  slope = alpha1
# phi_dot comes straight from the hardware gyro (GYR field).
# phi_ddot is the gradient of phi_dot with respect to time.

MIN_PHI_DEG = 2.0   # ignore samples too close to balance (noisy ratio)

# Compute phi_ddot via central differences on the raw gyro
phi_ddot = np.gradient(gyro, t)
phi_ddot = medfilt(phi_ddot, 5)   # 5-pt median filter — same approach as existing notebook

# Trim edge artefacts from gradient
sl = slice(2, -2)
phi_s    = phi[sl]
phi_dd_s = phi_ddot[sl]

# Only use samples where the car has actually fallen away from balance
mask = np.abs(phi_s) >= MIN_PHI_DEG
if mask.sum() < 5:
    print("WARNING: very few usable samples. Try releasing from a larger angle (15-20 deg off balance).")
    mask = np.ones_like(phi_s, dtype=bool)

slope, intercept, r_value, p_value, std_err = stats.linregress(phi_s[mask], phi_dd_s[mask])
alpha1 = slope

print("─" * 50)
print(f"  alpha1 (gravity term) : {alpha1:.4f}  [1/s²]")
print(f"  R²                    : {r_value**2:.3f}")
print(f"  Intercept             : {intercept:.3f}  (ideally ~0)")
print(f"  Equiv. pendulum L     : {9.81/alpha1:.3f} m  (large = high inertia, expected)")
print("─" * 50)
if r_value**2 < 0.8:
    print("WARNING: R² < 0.8 — try a larger release angle for a cleaner fit.")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('SysID Drop Test — alpha1 (gravity)', fontsize=13, fontweight='bold')

# 1. phi vs time
ax = axes[0, 0]
ax.plot(t, phi, color='steelblue', linewidth=2)
ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('φ = θ − θ_eq  (deg)')
ax.set_title('Pitch Deviation φ(t)')
ax.grid(True, linestyle='--', alpha=0.4)

# 2. gyro (phi_dot) vs time
ax = axes[0, 1]
ax.plot(t, gyro, color='darkorange', linewidth=2)
ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('φ̇  (deg/s)')
ax.set_title('Angular Rate φ̇(t)  [raw gyro]')
ax.grid(True, linestyle='--', alpha=0.4)

# 3. phi_ddot vs phi — the regression plot
ax = axes[1, 0]
ax.scatter(phi_s[mask], phi_dd_s[mask], s=8, alpha=0.5, color='steelblue', label='data')
phi_range = np.linspace(phi_s[mask].min(), phi_s[mask].max(), 100)
ax.plot(phi_range, alpha1 * phi_range, 'r-', linewidth=2,
        label=f'α₁·φ  (α₁ = {alpha1:.4f})')
ax.set_xlabel('φ  (deg)')
ax.set_ylabel('φ̈  (deg/s²)')
ax.set_title(f'Gravity SysID: φ̈ vs φ   R² = {r_value**2:.3f}')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

# 4. phi_ddot vs time: measured vs model
ax = axes[1, 1]
ax.plot(t, phi_ddot,      label='measured φ̈  (filtered)', alpha=0.7, linewidth=1.5)
ax.plot(t, alpha1 * phi,  label=f'α₁·φ (model)', color='red', linewidth=2, linestyle='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('φ̈  (deg/s²)')
ax.set_title('φ̈: measured vs gravity model')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('sysid_drop.png', dpi=150)
plt.show()
print("Saved → sysid_drop.png")

## Step B — Motor SysID  →  fit alpha2 (motor torque)

**Procedure:**
1. Configure gains (optional — use whatever you have been tuning with).
2. Send `START_PID` (cmd 10) to start a balancing session.
3. Let it run for several seconds so the motors are active.
4. Retrieve the log with `GET_RUN_DATA` (cmd 9).
5. Run the regression cell.

In [53]:
# ── CELL: Configure gains and start Phase A balancing ────────────────────────
KP = 10.0
KI = 0.0
KD = 1.0
SP = SETPOINT_DEG
S1 = 50.0   # process noise pitch
S2 = 50.0   # process noise rate
S3 = 1.0    # measurement noise

params = f"11|Kp:{KP}|Ki:{KI}|Kd:{KD}|SP:{SP}|S1:{S1}|S2:{S2}|S3:{S3}"
print(f"Sending gains: {params}")
ble.send_command(CommandTypes.SET_GAINS, params)
time.sleep(0.5)

raw_packets.clear()
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except: pass
ble.start_notify(ble.uuid['RX_STRING'], notification_handler)

print("Engaging Phase A: Place the car upright NOW...")
ble.send_command(CommandTypes.START_PID, "")

Sending gains: 11|Kp:10.0|Ki:0.0|Kd:1.0|SP:86.82|S1:50.0|S2:50.0|S3:1.0
Engaging Phase A: Place the car upright NOW...


In [ ]:
# ── CELL: Retrieve & parse balance run data ───────────────────────────────────
raw_packets.clear()
print("Requesting data (cmd 9)...")
ble.send_command(CommandTypes.GET_RUN_DATA, "")

# Poll until transfer completes OR packet stream goes idle.
timeout_s = 30.0
idle_window_s = 2.5
t_start = time.time()
last_count = 0
last_change = t_start

while time.time() - t_start < timeout_s:
    n = len(raw_packets)

    # Optional completion marker if firmware adds one later.
    if n > 0 and "complete" in raw_packets[-1].lower():
        break

    if n != last_count:
        last_count = n
        last_change = time.time()
    elif n > 0 and (time.time() - last_change) >= idle_window_s:
        break

    time.sleep(0.2)

print(f"Received {len(raw_packets)} packets.")
if raw_packets:
    print("First:", raw_packets[0])
    print("Last: ", raw_packets[-1])

# ── Parse ─────────────────────────────────────────────────────────────────────
times_raw_b, pitch_raw_b, kfpk_raw_b, motor_log_b, vel_raw_b, gyro_raw_b = [], [], [], [], [], []

for line in raw_packets:
    try:
        parts = {}
        for p in line.strip().split('|'):
            if ':' not in p:
                continue
            k, v = p.split(':', 1)
            parts[k] = float(v)

        if 'T' not in parts:
            continue

        times_raw_b.append(parts['T'])
        pitch_raw_b.append(parts.get('D1', 0.0))   # absolute theta (deg)
        kfpk_raw_b.append(parts.get('D2', 0.0))    # KF pitch/phi as logged in D2
        motor_log_b.append(parts.get('MOT', 0.0))  # signed PWM
        vel_raw_b.append(parts.get('VEL', 0.0))    # KF pitch-rate
        gyro_raw_b.append(parts.get('GYR', 0.0))   # raw gyro phi_dot (deg/s)
    except Exception:
        continue

if len(times_raw_b) == 0:
    print("No data received. Check BLE connection and that the robot ran.")
else:
    t_b = np.array([(x - times_raw_b[0]) / 1000.0 for x in times_raw_b])
    pitch_b = np.array(pitch_raw_b)
    phi_b = pitch_b - SETPOINT_DEG          # deviation from balance
    kfpk_b = np.array(kfpk_raw_b)
    mot_b = np.array(motor_log_b)
    u_b = mot_b / 255.0                     # normalized motor command [-1, 1]
    vel_b = np.array(vel_raw_b)
    gyro_b = np.array(gyro_raw_b)

    print(f"Parsed {len(t_b)} samples. Duration: {t_b[-1]:.2f} s")

In [ ]:
# ── CELL: Fit alpha2 (motor torque term) ─────────────────────────────────────
#
# Model:  phi_ddot = alpha1*phi - alpha2*u
# So:     phi_ddot - alpha1*phi = -alpha2*u
# Regression of (phi_ddot - alpha1*phi) on u gives slope = -alpha2.
#
# NOTE: alpha1 must be set from Step A above before running this cell.

MIN_ABS_U = 0.10   # ignore deadband samples where motors are barely on

# phi_ddot via gyro
phi_ddot_b = np.gradient(gyro_b, t_b)
phi_ddot_b = medfilt(phi_ddot_b, 5)

sl_b = slice(2, -2)
phi_s_b    = phi_b[sl_b]
phi_dd_s_b = phi_ddot_b[sl_b]
u_s_b      = u_b[sl_b]

residual = phi_dd_s_b - alpha1 * phi_s_b   # = -alpha2 * u

mask_u = np.abs(u_s_b) >= MIN_ABS_U
if mask_u.sum() < 5:
    print("WARNING: very few motor-active samples. Lower MIN_ABS_U or use a run with more motor activity.")

slope_u, intercept_u, r_u, p_u, se_u = stats.linregress(u_s_b[mask_u], residual[mask_u])
alpha2 = -slope_u

print("─" * 50)
print(f"  alpha2 (motor torque) : {alpha2:.4f}  [deg/s² per unit-u]")
print(f"  R²                    : {r_u**2:.3f}")
print(f"  Intercept             : {intercept_u:.3f}  (ideally ~0)")
print("─" * 50)
print()
print("  ── Paste into lab12_fixed.ino ──")
print(f"  const float KF_ALPHA1 = {alpha1:.4f}f;   // gravity term")
print(f"  const float KF_ALPHA2 = {alpha2:.4f}f;   // motor torque term")
if r_u**2 < 0.7:
    print()
    print("WARNING: R² < 0.7 — try a run where the motors are more active (lower Kp so the car oscillates).")

In [ ]:
# ── CELL: Motor SysID Plot ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Motor SysID — alpha2 (motor torque)', fontsize=13, fontweight='bold')

# 1. phi and u vs time
ax  = axes[0, 0]
ax2 = ax.twinx()
ax.plot(t_b, phi_b, label='φ  (deg)', color='steelblue', linewidth=1.8)
ax2.plot(t_b, u_b,  label='u  (norm)', color='tomato', alpha=0.6, linewidth=1.2)
ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('φ  (deg)')
ax2.set_ylabel('u  (normalised PWM)')
ax.set_title('φ and motor command u(t)')
ax.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

# 2. residual vs u — the regression plot
ax = axes[0, 1]
ax.scatter(u_s_b[mask_u], residual[mask_u], s=8, alpha=0.5, color='darkorange', label='data')
u_range = np.linspace(u_s_b[mask_u].min(), u_s_b[mask_u].max(), 100)
ax.plot(u_range, -alpha2 * u_range, 'r-', linewidth=2,
        label=f'−α₂·u  (α₂ = {alpha2:.4f})')
ax.set_xlabel('u  (normalised PWM)')
ax.set_ylabel('φ̈ − α₁φ  (deg/s²)')
ax.set_title(f'Motor SysID: residual vs u   R² = {r_u**2:.3f}')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

# 3. raw gyro vs KF pitch-rate
ax = axes[1, 0]
ax.plot(t_b, gyro_b, label='gyro φ̇  (raw)', alpha=0.7, linewidth=1.5)
ax.plot(t_b, vel_b,  label='KF φ̇', linewidth=2, linestyle='--', color='red')
ax.set_xlabel('Time (s)')
ax.set_ylabel('deg/s')
ax.set_title('Pitch Rate: raw gyro vs KF')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

# 4. raw pitch vs KF pitch (absolute)
ax = axes[1, 1]
ax.plot(t_b, pitch_b,                     label='θ raw  (DMP)', alpha=0.7, linewidth=1.5)
ax.plot(t_b, kfpk_b + SETPOINT_DEG,      label='KF θ', linewidth=2, linestyle='--', color='red')
ax.axhline(SETPOINT_DEG, color='k', linewidth=0.8, linestyle=':', label='setpoint')
ax.set_xlabel('Time (s)')
ax.set_ylabel('θ  (deg)')
ax.set_title('Pitch: raw vs KF')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('sysid_motor.png', dpi=150)
plt.show()
print("Saved → sysid_motor.png")

## Balance Run Overview

In [ ]:
# ── CELL: Balance run overview plot ──────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Balancing Run Overview', fontsize=13, fontweight='bold')

axes[0].plot(t_b, pitch_b,                label='θ raw  (DMP)', alpha=0.7, linewidth=1.5)
axes[0].plot(t_b, kfpk_b + SETPOINT_DEG, label='KF θ',          linewidth=2, linestyle='--', color='red')
axes[0].axhline(SETPOINT_DEG, color='k', linewidth=0.8, linestyle=':', label='setpoint')
axes[0].set_ylabel('θ  (deg)')
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.4)

axes[1].plot(t_b, gyro_b, label='gyro φ̇  (raw)', alpha=0.7, linewidth=1.5)
axes[1].plot(t_b, vel_b,  label='KF φ̇',           linewidth=2, linestyle='--', color='red')
axes[1].set_ylabel('φ̇  (deg/s)')
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle='--', alpha=0.4)

axes[2].plot(t_b, mot_b, color='darkgreen', linewidth=1.8, label='Motor PWM')
axes[2].axhline(0, color='k', linewidth=0.8)
axes[2].set_ylabel('PWM')
axes[2].set_xlabel('Time (s)')
axes[2].legend(fontsize=9)
axes[2].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('balance_run.png', dpi=150)
plt.show()
print("Saved → balance_run.png")

## Tips

| Issue | Fix |
|---|---|
| alpha1 R² < 0.8 | Release from a larger angle (15–20 deg off balance) so the car falls faster |
| alpha2 R² < 0.7 | Run a session with a lower `Kp` so the car oscillates rather than converging — more motor variation |
| KF still sluggish after updating alphas | Lower `S1`/`S2` from 50 toward 5–10 to trust the physics model more |
| KF overshoots measurement | Raise `S3` (measurement noise); DMP Quat6 RMS is typically 1–2 deg |
| GYR field is always 0 | Check that `myICM.getAGMT()` is being called in the `.ino` after `read_dmp_pitch()` |